In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from matplotlib import style
from matplotlib import pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

%matplotlib inline

style.use("fivethirtyeight")

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows의 경우
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

#### **1. 분석대상 그룹화하기**
##### **1.1 분석파일 코랩으로 마운트하기**


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
import pickle

# 2단계에서 확인한 정확한 파일 경로로 변경해주세요
file_path = '/content/drive/MyDrive/Mat-SME analysis-by-colab/0621_2025_sme_project_panel.pkl'

with open(file_path, 'rb') as f:
    data = pickle.load(f)

# 불러온 데이터 확인 (예시)
# print(data)

In [5]:
print(data.columns)

Index(['id', 'year', 'KEDCD', '결산기준일', '기업공개', '기업규모', '기업부설연구소유무', '기업상태',
       '기업유형', '기업형태', '도로명주소', '법인등록번호', '법인번호', '벤처인증발급일', '벤처인증유무',
       '사업자등록번호', '사업자번호', '설립일자', '소재부품전문기업만료일', '소재부품전문기업발급일', '소재부품전문기업유무',
       '업종명10차_세세분류', '업종코드10차_세세분류', '업체명', '업체명_형태포함', '연구개발전담부서유무',
       '이노비즈인증유무', '이노비즈인증유효시작일', '주요제품명', '매출액', '연구개발비', '국민연금가입자수',
       '고용보험가입자수', '자산총계', '부채총계', '자본총계', '유형자산', '무형자산', '영업이익손실',
       '당기순이익_재무', '매출총이익손실', '부가가치', '노동관계비용', '유동자산', '비유동자산', '유동부채',
       '비유동부채', '이자수익_손익', '영업외비용_이자비용', '자본금', '특허등록건수_등록일기준', '특허출원건수_출원일기준',
       '실용신안등록건수_등록일기준', '실용신안출원건수_출원일기준', '업종코드10차_대분류', '영업이익손실_수치',
       '이자비용_수치', '이자보상비율', '이자보상해석', '설립일자_dt', '업력', '소재지', 'project_data',
       'project_count', 'project_merged', 'project_sources', 'treated', '전체특허',
       '전체실용신안'],
      dtype='object')


##### **☆ 주요 참고사항**
###### project_data는 리스트 안에 딕셔너리들이 들어 있는 구조
###### 즉, [{dict}, {dict}, {dict}, ...] 형태

In [6]:
# 예를 들어 첫 번째 값을 확인하여 project_data가 어떤 형식인지 보니까
sample = data['project_data'].iloc[0]
print(sample)
print(type(sample))
# [로 시작하니 리스트이고 그 안에 바로 {가 따라오니 딕셔너리

[{'proj_명칭': {'사업명': '중소기업기술혁신개발', '사업구분': '일반연구개발사업', '사업_부처명': '중소벤처기업부', '내역사업명': '혁신형기업기술개발', '과제명': '성감염병 광대역 동시다중 분자진단 기술 개발', '__type__': 'unknown'}, 'proj_식별': {'이전과제고유번호': 'nan', '과제고유번호': '1425111657', '주관과제고유번호': '1425111657.0', '__type__': 'unknown'}, 'proj_관리수행': {'대표전문기관': '중소기업기술정보진흥원', '과제관리기관': '중소기업기술정보진흥원', '과제수행기관명': '(주)진매트릭스', '__type__': 'unknown'}, 'proj_기간': {'총연구기간-시작년월일': '2017-06-19', '총연구기간-종료년월일': '2019-06-18', '당해연구기간-시작년월일': '2017-06-19', '당해연구기간-종료년월일': '2018-06-18', '__type__': 'unknown'}, 'proj_주체': {'연구수행주체코드': 5, '연구수행주체': '중소기업', '__type__': 'unknown'}, 'proj_계속여부': {'계속과제여부구분코드': 1, '계속과제여부': '신규', '__type__': 'unknown'}, 'proj_연구단계': {'연구개발단계코드': 3, '연구개발단계': '개발연구', '__type__': 'unknown'}, 'proj_세부여부': {'세부과제지원코드': nan, '세부과제지원유형': None, '__type__': 'unknown'}, 'proj_지역': {'지역코드': 8, '지역': '경기도', '기초자치단체코드': 8131.0, '기초자치단체': '성남시', '__type__': 'unknown'}, 'proj_적용분야': {'경제사회목적코드': 7, '경제사회목적': '산업생산 및 기술', '적용분야코드1': 'Y09', '적용분야1': '제조업(의료, 정밀

In [7]:

# isinstance와 ast.literal_eval을 사용해서 좀 더 자세히 project_data의 구성요소를 보자

import ast

# 예시: 첫 번째 row의 구조 확인
raw_value = data['project_data'].iloc[0]

# 만약 문자열이라면 eval이 필요할 수 있음
if isinstance(raw_value, str):
    parsed = ast.literal_eval(raw_value)
else:
    parsed = raw_value

print(f"전체 타입: {type(parsed)}")  # <class 'list'>
print(f"첫 번째 요소 타입: {type(parsed[0])}")  # <class 'dict'>

# 첫 번째 딕셔너리의 key 예시 보기
print(parsed[0].keys())

전체 타입: <class 'list'>
첫 번째 요소 타입: <class 'dict'>
dict_keys(['proj_명칭', 'proj_식별', 'proj_관리수행', 'proj_기간', 'proj_주체', 'proj_계속여부', 'proj_연구단계', 'proj_세부여부', 'proj_지역', 'proj_적용분야', 'proj_연구내용', 'proj_예산', 'proj_주관공동', 'proj_인력'])


##### **1.2 분석파일 파악 및 전처리하기**


In [ ]:
# 불필요한 컬럼 드롭 -> 그닥 효용은 크지 않은 듯
# data = data.drop(columns=['전체실용신안', '실용신안등록건수_등록일기준', '실용신안출원건수_출원일기준'])


In [8]:
# 업종코드 대분류 생성
data['업종코드10차_대분류'] = (
    data['업종코드10차_세세분류']
    .str[1:3]  # 'C27199' -> '27'
    .apply(lambda x: int(x) if x.isdigit() else np.nan)
)

##### **분석 대상 정의**

In [10]:
treated_starts = data[data['treated'] == 1].groupby('id')['year'].min().reset_index()
treated_starts.columns = ['id', 'treated_firm_recrods_start_year']
print(treated_starts)

        id  treated_firm_recrods_start_year
0        1                             2017
1        2                             2017
2        3                             2017
3        4                             2017
4        5                             2017
...    ...                              ...
1982  1983                             2017
1983  1984                             2017
1984  1985                             2017
1985  1986                             2017
1986  1987                             2017

[1987 rows x 2 columns]


##### **처치여부 표시: 2019, 2020 과제**

In [ ]:
import pandas as pd
import ast
from collections import defaultdict

# 1. 내역사업명 키워드 (정확히 일치)
special_keywords = set([
    '2020년도 중소기업기술혁신개발사업 ‘시장대응형 과제’ 제2차 시행계획 공고',
    '소재부품기술기반혁신',
    '2020년도 중소기업기술혁신개발사업 \'시장대응형 과제\' 제1차 시행계획 수정공고',
    '소재부품패키지형',
    '2020년도 구매조건부신제품개발사업 구매연계형 과제 자유응모(2차) 시행계획 공고',
    '2020년도 중소기업기술혁신개발사업 ‘시장확대형 과제’ 제2차 시행계획 수정공고',
    '2020년도 중소기업기술혁신개발사업 ‘시장확대형 과제’ 제1차 시행계획 수정공고',
    '전략핵심소재자립화기술개발',
    '2020년도 구매조건부신제품개발사업 구매연계형 과제 자유응모(1차) 시행계획 공고',
    '소재부품이종기술융합형',
    '2020년도 창업성장기술개발사업 \'전략형 창업과제(소재·부품·장비)\' 제2차 시행계획 수정 공고',
    '2020년도 창업성장기술개발사업 ’전략형 창업과제(소재·부품·장비)‘ 제1차 시행계획 공고',
    '2020년도 중소기업기술혁신개발사업 시장확대형 과제 제4차 시행계획 공고(후불형과제)',
    '2020년도 구매조건부신제품개발사업 공동투자형 과제 자유응모(4차) 시행계획 공고',
    '2020년도 중소기업기술혁신개발사업 ‘시장확대형 과제’ 제3차 시행계획 공고(비대면 분야)',
    '2020년도 구매조건부신제품개발사업 공동투자형 과제 자유응모(7차) 시행계획 공고',
    '2020년도 구매조건부신제품개발사업 공동투자형 과제 자유응모(6차) 시행계획 공고',
    '2020년도 구매조건부신제품개발사업 공동투자형 과제 자유응모(3차) 시행계획 공고',
    '2020년도 구매조건부신제품개발사업 공동투자형 과제 자유응모(5차) 시행계획 공고',
    '2020년도 구매조건부신제품개발사업 공동투자형 과제 자유응모(2차) 시행계획 공고',
    '2020년도 구매조건부신제품개발사업 구매연계형 과제 지정공모(2차) 시행계획 공고',
    '이종기술융합형',
    '패키지형',
    '소재부품기술기반혁신사업',
    '2019년도 중소기업 기술혁신개발사업(소재·부품·장비분야 추경사업)',
    '2022년도 중소기업기술혁신개발사업 \'강소기업100\' 과제 시행계획 공고',
    '2022년도 중소기업기술혁신개발사업 \'강소기업100\' 과제 시행계획(2차) 공고',
    '2022년도 중소기업기술혁신개발사업 \'소부장전략\' 과제 시행계획 공고',
    '2022년도 중소기업기술혁신개발사업 ‘소부장일반(일반과제)’ 상반기 시행계획 공고',
    '2022년도 중소기업기술혁신개발사업 ‘소부장일반(일반과제)’ 하반기 시행계획 공고',
    '2022년도 중소기업기술혁신개발사업 ‘소부장일반(재도약과제)’ 상반기 시행계획 공고',
    '2022년도 중소기업기술혁신개발사업 소부장전략(함께달리기) 시행계획 공고',
    '2022년도 중소기업기술혁신개발사업(소부장전략) 대중소기업 상생모델 추천기업 사업계획서 접수 안내',
    '사업연계형(소부장 전략)',
    '중소기업기술혁신개발사업(소부장일반)(22년 컨소시엄형R&D 선정기업 대상)',
    '중소기업기술혁신개발사업(소부장전략)(22년 컨소시엄형R&D 선정기업 대상)',
    '2021년 중소기업기술혁신개발사업 \'강소기업100\' 과제 제1차 시행계획 공고',
    '2021년 중소기업기술혁신개발사업 \'소부장일반\' 과제 제1차 시행계획 수정 공고',
    '2021년 중소기업기술혁신개발사업 \'소부장전략 과제\' 제1차 시행계획 수정 공고',
    '2021년 중소기업기술혁신개발사업 소부장전략 과제 2차 시행계획 공고',
    '2021년도 중소기업기술혁신개발사업 강소기업100 과제 2차 시행계획 공고',
    '2021년도 중소기업기술혁신개발사업 소부장일반 과제 제 2차 시행계획 공고',
    '2021년도 중소기업기술혁신개발사업 소부장전략과제(함께달리기) 시행계획 공고',
    '2022년도 중소기업 구매조건부신제품개발사업 \'공동투자형 과제\' 자유공모(2차) 시행계획 공고',
    '2022년도 중소기업 구매조건부신제품개발사업 \'구매연계형 과제\' 자유공모(2차) 시행계획 공고',
    '2022년도 중소기업 구매조건부신제품개발사업 ‘공동투자형 과제’ 자유공모(1차) 시행계획  공고',
    '2022년도 중소기업 구매조건부신제품개발사업 ‘구매연계형 과제’ 자유공모(1차) 시행계획 공고',
    '2021년도 구매조건부신제품개발사업 공동투자형 과제 자유응모(2차) 시행계획 공고',
    '2021년도 구매조건부신제품개발사업 구매연계형 과제 자유응모(1차) 시행계획 공고',
    '2021년도 구매조건부신제품개발사업 구매연계형 과제 자유응모(2차) 시행계획 공고',
    '2022년도 창업성장기술개발사업 소재·부품·장비 스타트업100 연계과제 시행계획 공고',
    '2022년도 창업성장기술개발사업 전략형(소재부품장비) 제1차 시행계획 공고',
    '2021년도 창업성장기술개발사업 소부장 스타트업 100 연계과제 시행계획 공고',
    '2021년도 창업성장기술개발사업 전략형 과제(소재·부품·장비) 제1차 시행계획 공고',
    '2021년도 창업성장기술개발사업 전략형(소재·부품·장비) 제2차 시행계획 공고'
])

# 2. id별로 조건을 만족한 year 수집
from collections import defaultdict
id_to_special_years = defaultdict(set)

# 3. project_data를 순회하며 조건 검사
for _, row in data.iterrows():
    try:
        projects = ast.literal_eval(row['project_data']) if isinstance(row['project_data'], str) else row['project_data']
        for proj in projects:
            내역 = proj.get('proj_명칭', {}).get('내역사업명', '').strip()
            상태 = proj.get('proj_계속여부', {}).get('계속과제여부', '').strip()
            if 내역 in special_keywords and 상태 == '신규':
                id_to_special_years[row['id']].add(row['year'])
    except Exception:
        continue

# 4. special 컬럼: 조건 만족한 경우 year 반환
def assign_special(id_):
    years = id_to_special_years.get(id_, set())
    if years:
        return sorted(years)  # list 형태로 보여줌 (optional)
    return None

# 5. special1 컬럼: 2020이 있으면 2020, 아니면 None
def assign_special1(id_):
    years = id_to_special_years.get(id_, set())
    if 2020 in years:
        return 2020
    return None

# 6. 적용
data['special'] = data['id'].apply(assign_special)
data['special1'] = data['id'].apply(assign_special1)


In [ ]:
data[['id', 'year', 'special', 'special1']].to_csv("special1_상세결과.csv", index=False, encoding='utf-8-sig')


In [ ]:
# 전제: data에는 'year'와 'special1' 컬럼이 존재한다고 가정

# 1. post 기본값은 0으로 초기화
data['post'] = 0

# 2. 조건을 만족하는 경우 1로 설정
# special1이 NaN이 아닌 경우에만 비교 수행
data.loc[data['special1'].notna() & (data['year'] >= data['special1']), 'post'] = 1


In [ ]:
data.info()

#### **1.1 분석대상 그룹화 결과**

#### **2. 분석용 변수생성**

In [ ]:
import numpy as np

# 안전한 로그 변환 함수: 숫자로 변환 후 음수/결측 예외 처리
def safe_log(x):
    x = pd.to_numeric(x, errors='coerce')  # 문자열 포함 시 NaN 처리
    return np.log1p(np.where(x < 0, 0, x))

# 주요 재무 변수 로그 변환
cols_to_log = {
    'log_revenue': '매출액',
    'log_gpl': '매출총이익손실',
    'log_oinl': '영업이익손실',
    'log_np': '당기순이익_재무',
    'log_assets': '자산총계',
    'log_equity': '자본총계',
    'log_debt': '부채총계',
    'log_ca': '유동자산',
    'log_nca': '비유동자산',
    'log_cl': '유동부채',
    'log_ncl': '비유동부채',
    'log_cap': '자본금',
    'log_icr': '이자보상비율',
    'log_employment1': '고용보험가입자수',
    'log_employment2': '국민연금가입자수',
    'log_laborcost': '노동관계비용',
    'log_rnd': '연구개발비',
    'log_age': '업력'
}

for new_col, original_col in cols_to_log.items():
    data[new_col] = safe_log(data[original_col])


In [ ]:
# 주요변수 중 특허 이진화
data['has_patent'] = data['전체특허'].fillna(0).apply(lambda x: 1 if x > 0 else 0)

# 2. log 변환한 특허 수 변수 생성 (log(특허+1))
data['log_patent_count'] = np.log1p(data['전체특허'])

In [ ]:
# 1. treated == 1 조건에 해당하는 행 필터링
treated_df = data[data['treated'] == 1]

# 2. 원하는 컬럼만 선택하여 새로운 데이터프레임 생성
columns_to_select = [
    'id', 'year', 'special1', 'post',
    'log_rnd', 'log_revenue', 'log_assets', 'log_employment1'
]
treated_selected = treated_df[columns_to_select]


In [ ]:
treated_selected

#### **2.1 기술통계량**

In [ ]:
# 집단별 주요 변수 요약 통계량
summary_cols = ['매출액', '자산총계', '자본총계', '부채총계', '고용보험가입자수',
               '노동관계비용', '연구개발비', '업력', '전체특허']
summary_stats = data.groupby('group')[summary_cols].describe().round(1)
print("\n[집단별 주요 변수 요약 통계]")
print(summary_stats)

# 로그 변환된 변수에 대해서도 요약
log_cols = ['log_revenue', 'log_assets', 'log_equity', 'log_debt', 'log_employment1',
            'log_laborcost', 'log_rnd', 'log_age', 'has_patent']
if set(log_cols).issubset(data.columns):
    log_summary = data.groupby('group')[log_cols].describe().round(2)
    print("\n[로그 변환 변수 요약 통계]")
    print(log_summary)

summary_stats.to_csv('집단별주요변수_자연수_v2.csv', index=False, encoding='utf-8-sig')
log_summary.to_csv('집단별주요변수_로그_v2.csv', index=False, encoding='utf-8-sig')

#### **3. 정책효과 분석**

##### **Double DiD**

In [ ]:
import pandas as pd

# 1. special1이 2020인 id를 뽑음
ids_with_2020 = (
    treated_selected.loc[pd.to_numeric(treated_selected['special1'], errors='coerce') == 2020, 'id']
    .unique()
)

# 2. id가 위에 포함되면 t_group = 1
treated_selected['t_group'] = treated_selected['id'].isin(ids_with_2020).astype(int)

In [ ]:
# whether treatment has occured at all
treated_selected['after'] = treated_selected['year'] >= 2020
# whether it has occurred to this entity
treated_selected['treatafter'] = treated_selected['after'] * treated_selected['t_group']

# Step 3:

ax = treated_selected.pivot_table(
    index='year',
    columns='t_group',
    values='log_rnd',
    aggfunc='mean'  # 중복 있는 경우 평균값 사용
).plot(
    figsize=(20, 10),
    marker='.',
    markersize=20,
    title='R&D and Time',
    xlabel='Year',
    ylabel='log R&D',
    xticks=treated_selected['year'].drop_duplicates().sort_values().astype(int)
)

ax.axvline(x=2020, color='gray', linestyle='--')
ax.legend(loc='upper left', title='treat', prop={'size': 20})


# Step 4:

# # statsmodels has two separate APIs
# # the original API is more complete both in terms of functionality and documentation
# step 1: 필요한 열만 추출하고 NaN/inf 제거
X_cols = ['t_group', 'treatafter', 'after', 'log_revenue', 'log_assets', 'log_employment1']
treated_clean = treated_selected[X_cols + ['log_rnd']].replace([np.inf, -np.inf], np.nan).dropna()

# step 2: float으로 명확하게 변환
treated_clean = treated_clean.astype(float)

# step 3: 변수 설정
X = sm.add_constant(treated_clean[X_cols])
y = treated_clean['log_rnd']

# step 4: 회귀 실행
sm_fit = sm.OLS(y, X).fit()

# the formula API is more familiar for R users
# it can be accessed through an alternate constructor bound to each model class
smff_fit = sm.OLS.from_formula('log_rnd ~ 1 + t_group + treatafter + after + log_revenue + log_assets + log_employment1', data=treated_selected).fit()

# it can also be accessed through a separate namespace
import statsmodels.formula.api as smf
smf_fit = smf.ols('log_rnd ~ 1 + t_group + treatafter + after + log_revenue + log_assets + log_employment1', data=treated_selected).fit()

# if using jupyter, rich output is displayed without the print function
# we should see three identical outputs
print(sm_fit.summary())
print(smff_fit.summary())
print(smf_fit.summary())

##### **Panel OLS**

In [ ]:
treated_selected['time_to_treat'] = (
    treated_selected['year'].sub(treated_selected['special1'])
    .fillna(0).astype('int'))

treated_selected = (
    pd.get_dummies(treated_selected, columns=['time_to_treat'], prefix='INX')
      .rename(columns=lambda x: x.replace('-', 'm'))
      .drop(columns='INX_m1')
      .set_index(['id', 'year'])
)

scalars = ['log_revenue', 'log_assets', 'log_employment1']
factors = treated_selected.columns[treated_selected.columns.str.contains('INX')]
exog = factors.union(scalars)
endog = 'log_rnd'

import pandas as pd
import linearmodels as lm

mod = lm.PanelOLS(treated_selected[endog],
                  treated_selected[exog],
                  entity_effects=True, time_effects=True)
fit = mod.fit(cov_type='clustered', cluster_entity=True)
fit.summary

In [ ]:
inxnames = treated_selected.columns[range(13,treated_selected.shape[1])]
formula = '{} ~ {} + EntityEffects + TimeEffects'.format(endog, '+'.join(exog))

mod = lm.PanelOLS.from_formula(formula,treated_selected)

clfe = mod.fit(cov_type = 'clustered',
    cluster_entity = True)

clfe.summary

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Get coefficients and CIs
res = pd.concat([clfe.params, clfe.std_errors], axis = 1)

res['ci'] = res['std_error']*1.96


res = res.filter(like='INX', axis=0)

res.index = (
    res.index
        .str.replace('INX_', '')
        .str.replace('m', '-')
        .astype('int')
        .rename('time_to_treat')
)

res.reindex(range(res.index.min(), res.index.max()+1)).fillna(0)

# 1. 기존 결과 복사
res_plot = res.copy()

# 2. 기준 시점 -1을 강제로 추가 (값은 0)
res_plot.loc[-1] = {'parameter': 0, 'std_error': 0, 'ci': 0}

# 3. 정렬 및 결측 포함한 전체 시점 만들기
res_plot = (
    res_plot
    .reindex(range(res_plot.index.min(), res_plot.index.max() + 1))
    .reset_index()
    .rename(columns={'index': 'time_to_treat'})
)

# 4. 결측이 아닌 값만 시각화 (선 없이 점만)
res_clean = res_plot.dropna(subset=['parameter'])

plt.errorbar(
    res_clean['time_to_treat'],
    res_clean['parameter'],
    yerr=res_clean['ci'],
    fmt='o',            # 점만 (선 없이)
    capsize=4,
    color='tab:blue'
)

# 기준선 추가
plt.axhline(0, linestyle='dashed', color='gray')
plt.axvline(0, linestyle='dashed', color='gray')

# 라벨
plt.xlabel('Time to Treatment')
plt.ylabel('Estimated Effect')
plt.title('Event Study')

plt.grid(True)
plt.tight_layout()
plt.show()


##### **다른 형태의 그래프**

In [ ]:
# 기존 reindex
res_plot = (
    res.reindex(range(res.index.min(), res.index.max() + 1))
       .reset_index()
)

# -1 시점이 NaN이면 직접 값 추가 (parameter = 0, ci = 0)
if -1 in res_plot['time_to_treat'].values:
    res_plot.loc[res_plot['time_to_treat'] == -1, ['parameter', 'ci']] = [0, 0]
else:
    res_plot = pd.concat([
        pd.DataFrame({'time_to_treat': [-1], 'parameter': [0], 'ci': [0]}),
        res_plot
    ], ignore_index=True).sort_values('time_to_treat')

# 다시 인덱스 정렬
res_plot = res_plot.sort_values('time_to_treat')

# 그리기
ax = res_plot.plot(
    x='time_to_treat',
    y='parameter',
    yerr='ci',
    legend=False,
    linestyle='-',
    marker='o',
    color='tab:blue'
)

# 기준선 추가
ax.axhline(0, linestyle='dashed', color='gray')
ax.axvline(0, linestyle='dashed', color='gray')

# 라벨 및 제목
ax.set_xlabel('Time to Treatment')
ax.set_ylabel('Estimated Effect')
ax.set_title('Event Study: Effect Over Time')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. 시점 범위 생성
tt_range = range(res.index.min(), res.index.max() + 1)

# 2. res 재정렬 + -1 시점 보정
res_plot = res.reindex(tt_range).copy()

# 3. -1 시점 값이 없다면 0으로 지정
if -1 not in res_plot.index or pd.isna(res_plot.loc[-1, 'parameter']):
    res_plot.loc[-1, 'parameter'] = 0
    res_plot.loc[-1, 'ci'] = 0

# 4. 선 그리기
plt.plot(res_plot.index, res_plot['parameter'], marker='o', color='tab:blue', label='Estimate')
plt.fill_between(
    res_plot.index,
    res_plot['parameter'] - res_plot['ci'],
    res_plot['parameter'] + res_plot['ci'],
    color='tab:blue',
    alpha=0.2,
    label='95% CI'
)

# 5. 기준선 추가
plt.axhline(0, linestyle='dashed', color='gray')
plt.axvline(0, linestyle='dashed', color='gray')

# 6. 라벨
plt.xlabel('Time to Treatment')
plt.ylabel('Estimated Effect')
plt.title('Event Study Plot with -1 Set to 0')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# **ground**

#### **코드1**
#### **기업규모와 상태(# 3)에 따른 데이터 분류**

이 코드 스니펫은 초기 `data` DataFrame을 대상으로 다음과 같은 세 가지 주요 필터링 작업을 순차적으로 적용하여 분석에 필요한 특정 기업 그룹을 정의합니다.

1.  **산업 분야 제한**: `'업종코드10차_세세분류'`가 'C'로 시작하는 기업들만 선택하여 특정 산업군(예: 제조업)에 해당하는 기업들을 걸러냅니다.
2.  **데이터의 시간적 완전성 확보**: 2017년부터 2023년까지 7개 연도에 걸친 모든 데이터가 존재하는 기업들만 선택하여, 시계열 분석의 일관성을 보장하고 결측 연도로 인한 편향을 방지합니다.
3.  **기업 특성(규모 및 상태) 제한**: `'기업규모'`가 '중기업', '소기업', '한시성중소기업', '중견기업', '보호대상중견기업' 중 하나이고, `'기업상태'`가 '정상'인 기업들만 선택하여, 연구 대상 기업의 특성을 명확히 하고 안정적인 기업들을 분석에 포함시킵니다.

이러한 필터링 과정을 통해, `data` DataFrame은 특정 산업 분야에 속하며, 일관된 시간 범위의 데이터를 가지고 있고, 정의된 규모 및 상태 기준을 만족하는 기업들로 구성된 정제된 데이터셋으로 변환됩니다. 이는 이후의 통계 분석이나 모델링에 적합한 형태로 데이터를 준비하는 중요한 전처리 단계입니다.

In [ ]:
# 1. 업종코드 필터링
data = data[data['업종코드10차_세세분류'].astype(str).str.startswith('C')]

# 2. 연도 커버리지 필터
valid_ids = data.groupby('id')['year'].agg(lambda x: set(range(2017, 2024)).issubset(set(x)))
data = data[data['id'].isin(valid_ids[valid_ids].index)]

# 3. 기업규모 + 기업상태 추가 조건
valid_sizes = ['중기업', '소기업', '한시성중소기업',
              '중견기업', '보호대상중견기업'
              ]
valid_statuses = ['정상']

data = data[data['기업규모'].isin(valid_sizes)]
data = data[data['기업상태'].isin(valid_statuses)]

### **코드 해설**

#### **1. 업종코드 필터링 (`data = data[data['업종코드10차_세세분류'].astype(str).str.startswith('C')]`)**

*   **기능**: 데이터프레임 `data`에서 `'업종코드10차_세세분류'` 컬럼의 값이 'C'로 시작하는 행들만 필터링하여 남깁니다.
*   **메서드/클래스/라이브러리**:
    *   `pandas.DataFrame.astype(str)`: DataFrame의 컬럼(`Series`)을 문자열(string) 타입으로 변환합니다. 이는 해당 컬럼에 대해 문자열 관련 작업을 수행하기 위한 필수적인 전처리 과정입니다.
    *   `pandas.Series.str.startswith('C')`: 문자열 Series에 적용되는 Pandas의 문자열 접근자(`.str`)를 사용하여 각 요소가 'C'로 시작하는지 여부를 True/False로 반환합니다.
    *   `data[...]`: 불리언 인덱싱(Boolean Indexing)을 사용하여 `True`에 해당하는 행들만 선택합니다.

#### **2. 연도 커버리지 필터 (`valid_ids = data.groupby('id')['year'].agg(lambda x: set(range(2017, 2024)).issubset(set(x)))`)**

*   **기능**: 2017년부터 2023년까지의 모든 연도 데이터를 가지고 있는 `id` (기업)만 선택하여 `data`프레임을 필터링합니다.
*   **메서드/클래스/라이브러리**:
    *   `pandas.DataFrame.groupby('id')`: `data`프레임을 `'id'` 컬럼을 기준으로 그룹화합니다.
    *   `pandas.core.groupby.DataFrameGroupBy['year']`: 그룹화된 객체에서 `'year'` 컬럼을 선택합니다.
    *   `pandas.core.groupby.SeriesGroupBy.agg(lambda x: ...)`: 각 그룹에 대해 람다(lambda) 함수로 정의된 집계(aggregation) 작업을 수행합니다.
        *   `lambda x: set(range(2017, 2024)).issubset(set(x))`: 익명 함수로, 각 `id` 그룹의 `year` 데이터(Series `x`)를 `set`으로 변환한 뒤, `set(range(2017, 2024))` (즉, {2017, 2018, ..., 2023})가 그 `id`가 가진 연도 집합의 부분집합(`issubset`)인지 확인합니다.
        *   `set()`: Python의 내장 함수로, 중복되지 않는 요소들의 순서 없는 컬렉션인 `set`을 생성합니다.
        *   `range(2017, 2024)`: Python의 내장 함수로, 2017부터 2023까지의 정수를 생성합니다.
        *   `set.issubset()`: `set` 객체의 메서드로, 다른 `set`의 부분집합인지 여부를 확인합니다.
    *   `valid_ids[valid_ids].index`: `valid_ids` Series에서 값이 `True`인 인덱스(`id`)만 추출합니다.
    *   `data[data['id'].isin(...)]`: `data`프레임에서 `'id'` 컬럼의 값이 `valid_ids`에 포함되는 행들만 필터링합니다. `pandas.Series.isin()`은 Series의 각 요소가 주어진 값들의 컬렉션에 포함되는지 확인합니다.

#### **3. 기업규모 + 기업상태 추가 조건 필터링 (`data = data[data['기업규모'].isin(valid_sizes)]` 등)**

*   **기능**: `data`프레임에서 `'기업규모'` 컬럼의 값이 `valid_sizes` 리스트에 포함되고, `'기업상태'` 컬럼의 값이 `valid_statuses` 리스트에 포함되는 행들만 선택합니다.
*   **메서드/클래스/라이브러리**:
    *   `valid_sizes = [...]`, `valid_statuses = [...]`: Python의 내장 데이터 타입인 `list`를 사용하여 필터링 기준이 되는 값들을 정의합니다.
    *   `data['기업규모'].isin(valid_sizes)` 및 `data['기업상태'].isin(valid_statuses)`: 앞에서 설명한 `pandas.Series.isin()` 메서드를 사용하여 각 컬럼의 값이 특정 리스트에 포함되는지 여부를 True/False로 반환합니다.
    *   `data[...]`: 불리언 인덱싱을 사용하여 `True`에 해당하는 행들만 선택합니다.

#### **코드2**
#### **딕셔너리로 채워진 리스트는 몇 개인지, 비어있는 리스트는 몇 개인지 파악**

이 코드 스니펫은 `data` DataFrame의 `project_data` 컬럼에 저장된 복잡한 데이터 구조(예: 리스트 안의 딕셔너리)를 분석하여, 해당 컬럼의 모든 행이 어떤 형태의 데이터 구조를 가지고 있는지 요약 통계를 제공합니다. 예를 들어, `list[empty]` (비어있는 리스트)가 몇 개 있는지, `list[dict]` (딕셔너리로 채워진 리스트)가 몇 개 있는지 등을 파악할 수 있어 데이터 전처리나 분석 방향을 설정하는 데 유용합니다.

In [ ]:
# inspector

def inspect_structure(row):
    try:
        obj = ast.literal_eval(row) if isinstance(row, str) else row
        if isinstance(obj, list):
            if not obj:
                return "list[empty]"
            return f"list[{type(obj[0]).__name__}]"
        return type(obj).__name__
    except Exception as e:
        return f"Error: {e}"

data['project_data'].apply(inspect_structure).value_counts()

### **코드 해설**

#### **1. `inspect_structure` 함수 정의**

*   **기능**: DataFrame의 한 행에서 `project_data`와 같은 복잡한 구조의 데이터를 받아 그 내부 구조를 문자열로 반환하는 함수입니다. 주로 리스트 안에 딕셔너리가 있는 형태를 파악하는 데 중점을 둡니다.
*   **메서드/클래스/라이브러리**:
    *   `ast` 라이브러리: Python의 추상 구문 트리(Abstract Syntax Tree)를 다루는 모듈입니다. 여기서는 `ast.literal_eval()` 메서드를 사용하여 안전하게 문자열 형태의 Python 리터럴(리스트, 딕셔너리 등)을 실제 Python 객체로 변환합니다. 이는 악의적인 코드가 실행될 위험 없이 문자열을 평가할 수 있게 해줍니다.
    *   `isinstance(obj, type)`: `obj`가 `type`의 인스턴스(객체)인지 확인하여 `True` 또는 `False`를 반환하는 Python 내장 함수입니다. 여기서는 `obj`가 리스트(`list`)인지 확인하는 데 사용됩니다.
    *   `type(obj).__name__`: 객체 `obj`의 타입(클래스) 이름을 문자열로 가져옵니다. 예를 들어, 딕셔너리라면 `'dict'`를 반환합니다.
    *   `try-except` 블록: `ast.literal_eval()` 과정이나 다른 처리 과정에서 발생할 수 있는 오류를 안전하게 처리하기 위한 구문입니다. 오류가 발생하면 `'Error: {e}'` 형태의 문자열을 반환합니다.

#### **2. `data['project_data'].apply(inspect_structure).value_counts()`**

*   **기능**: `inspect_structure` 함수를 `data` DataFrame의 `project_data` 컬럼의 각 행에 적용하고, 그 결과로 반환된 구조 문자열들의 빈도를 계산합니다.
*   **메서드/클래스/라이브러리**:
    *   `pandas.Series.apply(func)`: Pandas Series의 각 요소에 대해 지정된 함수(`func`)를 적용합니다.
    *   `pandas.Series.value_counts()`: Series 내의 고유한 값들과 각 값의 출현 빈도를 계산하여 새로운 Series로 반환합니다. 기본적으로 빈도수가 높은 순서로 정렬됩니다.

#### **코드3**
#### 각 기업-연도 관측치에 대해 수행된 프로젝트의 총 개수

이 코드는 `data` DataFrame 내의 `project_data` 컬럼을 분석하여, 각 기업-연도 관측치에 대해 수행된 프로젝트의 총 개수를 계산합니다. 이 개수는 새로운 컬럼 `project_data_count`에 저장됩니다. 이를 통해 각 기업이 특정 연도에 몇 개의 프로젝트를 진행했는지 쉽게 파악할 수 있으며, 이는 기업의 활동성이나 연구개발 노력 등을 측정하는 지표로 활용될 수 있습니다.

In [ ]:
def count_dicts(row):
    try:
        obj = ast.literal_eval(row) if isinstance(row, str) else row
        if isinstance(obj, list):
            return len(obj)
        return 0
    except:
        return 0

data['project_data_count'] = data['project_data'].apply(count_dicts)

data['project_data_count']

### **코드 해설**

#### **1. `count_dicts` 함수 정의**

*   **기능**: 입력받은 `row`가 유효한 Python 리스트(특히 딕셔너리를 포함하는)의 문자열 표현이거나 이미 리스트인 경우, 해당 리스트 내의 요소(딕셔너리) 개수를 반환합니다. 오류 발생 시 또는 리스트가 아닌 경우 0을 반환하여 안전하게 처리합니다.
*   **메서드/클래스/라이브러리**:
    *   `ast` 라이브러리: Python의 추상 구문 트리(Abstract Syntax Tree)를 다루는 모듈입니다. 여기서는 `ast.literal_eval(row)` 메서드를 사용합니다. 이는 문자열 형태의 Python 리터럴(예: `"[{'key': 'value'}]"`와 같은 문자열)을 실제 Python 객체(리스트, 딕셔너리 등)로 안전하게 변환합니다. `eval()` 함수와 달리 악의적인 코드가 실행될 위험이 적습니다.
    *   `isinstance(obj, str)`: `obj`가 문자열 타입인지 확인합니다. `project_data` 컬럼의 데이터가 문자열로 저장되어 있을 수 있기 때문에 이를 실제 객체로 변환하기 위해 사용됩니다.
    *   `isinstance(obj, list)`: `obj`가 리스트 타입인지 확인합니다. 이 함수는 리스트인지 여부를 True/False로 반환합니다.
    *   `len(obj)`: Python 내장 함수로, 리스트 `obj`의 길이를 반환합니다. 즉, 리스트 안에 몇 개의 요소(여기서는 딕셔너리)가 있는지 세는 역할을 합니다.
    *   `try-except` 블록: `ast.literal_eval()` 과정이나 다른 데이터 처리 과정에서 발생할 수 있는 오류를 안전하게 처리합니다. 예를 들어, `project_data` 컬럼에 예상치 못한 형식의 데이터가 있거나 손상된 데이터가 있을 경우 오류 대신 0을 반환하도록 합니다.

#### **2. `data['project_data'].apply(count_dicts)`**

*   **기능**: 정의된 `count_dicts` 함수를 `data` DataFrame의 `'project_data'` 컬럼에 있는 각 요소에 적용합니다. 이렇게 하면 각 행의 `project_data`에 포함된 프로젝트(딕셔너리)의 개수가 계산됩니다.
*   **메서드/클래스/라이브러리**:
    *   `pandas.Series.apply(func)`: Pandas Series의 각 요소에 대해 지정된 함수(`func`)를 적용합니다. 이 경우 `count_dicts` 함수가 `project_data` 컬럼의 모든 행에 대해 호출됩니다.

#### **3. `data['project_data_count'] = ...`**

*   **기능**: `apply` 메서드의 결과로 반환된 Series(각 `project_data` 항목의 딕셔너리 개수)를 `data` DataFrame의 새로운 컬럼인 `'project_data_count'`에 할당합니다.

#### **4. `data['project_data_count']`**

*   **기능**: 새로 생성된 `project_data_count` 컬럼의 내용을 출력합니다. 이는 계산 결과가 어떻게 저장되었는지 확인하는 용도입니다.

#### **코드4**
#### '사업명'에 '소재부품', '내역사업명'에 '패키지형'이 포함되고 '19년 또는 '20년에 수행된 신규 과제에 대한 정보를 추출, 집계

이 코드는 `data` DataFrame에서 특정 조건을 만족하는 정부 연구개발 프로젝트(즉, '사업명'에 '소재부품', '내역사업명'에 '패키지형'이 포함되고 2019년 또는 2020년에 수행된 신규 과제)에 대한 정보를 추출하고 집계하는 과정을 자동화합니다. 최종적으로 각 기업이 해당 기간 동안 특정 내역사업명을 통해 수주한 정부 연구비 총액을 연도, 내역사업명, 기업 ID별로 정리하여 CSV 파일로 출력합니다. 이는 특정 유형의 R&D 지원 정책이 기업의 연구 활동에 미치는 영향을 분석하는 데 활용될 수 있는 중요한 전처리 단계입니다.

In [ ]:
## 조건부합 과제 정보 추출

import ast
import pandas as pd

def extract_government_funding(row):
    try:
        obj = ast.literal_eval(row) if isinstance(row, str) else row
        result = []
        for proj in obj:
            proj_name = proj.get('proj_명칭', {})
            사업명 = str(proj_name.get('사업명', ''))
            내역사업명 = str(proj_name.get('내역사업명', ''))

            # 조건: '소재부품' + '패키지형'
            if '소재부품' in 사업명 and '패키지형' in 내역사업명:
                proj_budget = proj.get('proj_예산', {})
                정부연구비 = proj_budget.get('정부연구비(원)', 0)
                result.append({
                    '사업명': 사업명,
                    '내역사업명': 내역사업명,
                    '정부연구비(원)': 정부연구비
                })
        return result if result else None
    except:
        return None

# 1. 조건에 맞는 행 필터링
def row_has_target_project(row):
    try:
        obj = ast.literal_eval(row) if isinstance(row, str) else row
        for proj in obj:
            proj_name = proj.get('proj_명칭', {})
            if '소재부품' in str(proj_name.get('사업명', '')) and \
               '패키지형' in str(proj_name.get('내역사업명', '')):
                return True
        return False
    except:
        return False

# 2. 필터링 적용
filtered = data[data['project_data'].apply(row_has_target_project)]

# 3. 대상 연도만 필터링
filtered = filtered[filtered['year'].isin([2019, 2020])]

# 4. 정부연구비 정보 추출
expanded_rows = []

for idx, row in filtered.iterrows():
    extracted = extract_government_funding(row['project_data'])
    if extracted:
        for item in extracted:
            expanded_rows.append({
                'id': row['id'],
                'year': row['year'],
                '내역사업명': item['내역사업명'],
                '정부연구비(원)': item['정부연구비(원)']
            })

# 5. 결과 DataFrame으로 정리
result_df = pd.DataFrame(expanded_rows)

# 6. 연도-내역사업명-id 별로 확인
grouped = result_df.groupby(['year', '내역사업명', 'id'])['정부연구비(원)'].sum().reset_index()

# 7. 결과를 출력
print(grouped)

### **코드 해설**

#### **1. `extract_government_funding` 함수 정의**

*   **기능**: `project_data` 컬럼의 각 항목(리스트 안의 딕셔너리)을 순회하며, `'사업명'`에 '소재부품'이, `'내역사업명'`에 '패키지형'이 포함된 프로젝트의 `'정부연구비(원)'`을 추출하여 리스트 형태로 반환합니다. 해당 조건에 맞는 프로젝트가 없거나 처리 중 오류가 발생하면 `None`을 반환합니다.
*   **메서드/클래스/라이브러리**:
    *   `ast.literal_eval(row)`: 문자열로 저장된 Python 리터럴(여기서는 리스트 형태)을 실제 Python 객체로 안전하게 변환합니다.
    *   `isinstance(row, str)`: 입력 `row`가 문자열인지 확인합니다.
    *   `dict.get(key, default_value)`: 딕셔너리에서 `key`에 해당하는 값을 가져오되, `key`가 없으면 `default_value`를 반환하여 오류를 방지합니다.
    *   `str.get('사업명', '')`: 사업명이 없을 경우 빈 문자열을 반환하여 `in` 연산을 안전하게 수행할 수 있게 합니다.
    *   `'키워드' in 문자열`: 문자열 안에 특정 키워드가 포함되어 있는지 확인합니다.
    *   `list.append()`: 리스트에 요소를 추가합니다.
    *   `try-except` 블록: 데이터 처리 중 발생할 수 있는 예외(오류)를 처리합니다.

#### **2. `row_has_target_project` 함수 정의**

*   **기능**: `project_data` 컬럼의 한 행(row)을 받아, 해당 행의 프로젝트 정보 리스트 안에 `'사업명'`에 '소재부품'이, `'내역사업명'`에 '패키지형'이 포함된 프로젝트가 하나라도 있는지 여부를 `True`/`False`로 반환합니다. 이는 필터링을 위한 보조 함수입니다.
*   **메서드/클래스/라이브러리**: `extract_government_funding` 함수와 유사한 메서드들을 사용하며, 주로 불리언 값을 반환하여 조건을 만족하는 행을 식별하는 데 초점을 맞춥니다.

#### **3. 필터링 적용 (`filtered = data[data['project_data'].apply(row_has_target_project)]` 외)**

*   **기능**: 전체 `data` DataFrame에서 `row_has_target_project` 함수를 사용하여 특정 조건을 만족하는 프로젝트를 포함하는 행들만 1차 필터링합니다. 이어서 `year` 컬럼 값이 2019 또는 2020인 행들로 2차 필터링을 수행하여 분석 대상을 좁힙니다.
*   **메서드/클래스/라이브러리**:
    *   `pandas.Series.apply()`: Pandas Series의 각 요소에 함수를 적용합니다.
    *   불리언 인덱싱 (`DataFrame[boolean_series]`): `True`/`False` 값을 가진 Series를 사용하여 DataFrame의 행을 선택합니다.
    *   `pandas.Series.isin([value1, value2])`: Series의 각 요소가 지정된 값들 중 하나에 포함되는지 여부를 확인합니다.

#### **4. 정부 연구비 정보 추출 및 DataFrame 생성 (`expanded_rows`, `result_df` 부분)**

*   **기능**: 필터링된 `filtered` DataFrame의 각 행에 대해 `extract_government_funding` 함수를 호출하여 조건을 만족하는 프로젝트의 상세 정보를 추출합니다. 추출된 정보는 `expanded_rows` 리스트에 쌓이고, 최종적으로 `pandas.DataFrame`으로 변환하여 `result_df`를 생성합니다.
*   **메서드/클래스/라이브러리**:
    *   `DataFrame.iterrows()`: DataFrame의 각 행을 (인덱스, Series) 쌍으로 순회합니다.
    *   `pd.DataFrame(list_of_dicts)`: 딕셔너리 리스트를 Pandas DataFrame으로 변환합니다.

#### **5. 데이터 집계 및 저장 (`grouped`, `to_csv` 부분)**

*   **기능**: `result_df`를 `'year'`, `'내역사업명'`, `'id'` 기준으로 그룹화한 후, 각 그룹별로 `'정부연구비(원)'`의 합계를 계산합니다. 이 집계된 결과는 `grouped` DataFrame으로 저장되고, 최종적으로 "정부연구비_소재부품_패키지형.csv" 파일로 저장됩니다.
*   **메서드/클래스/라이브러리**:
    *   `DataFrame.groupby(['col1', 'col2'])['target_col'].sum()`: 지정된 컬럼들을 기준으로 그룹화하고, `target_col`의 합계를 계산합니다.
    *   `.reset_index()`: `groupby` 결과로 생성된 MultiIndex를 일반 컬럼으로 변환합니다.
    *   `DataFrame.to_csv()`: DataFrame을 CSV 파일로 저장합니다. `index=False`는 DataFrame 인덱스를 CSV에 포함하지 않도록 하고, `encoding='utf-8-sig'`는 한글 깨짐을 방지하는 인코딩 방식입니다.

#### **코드5**
#### `project_data` '소재부품', '소부장' 등 을 포함하는 정부 연구개발 사업명을 식별하고, 해당 사업명을 가진 기업-연도 관측치들을 필터링

이 코드는 `data` DataFrame의 `project_data` 컬럼에서 특정 키워드(예: '소재부품', '소부장' 등)를 포함하는 정부 연구개발 사업명을 식별하고, 해당 사업명을 가진 기업-연도 관측치들을 필터링합니다. 최종적으로 필터링된 데이터에서 발견된 모든 고유한 사업명의 목록과 그 개수를 출력하여, 특정 정책 분야와 관련된 프로젝트들의 분포를 파악하는 데 도움을 줍니다. 이는 R&D 정책 효과 분석의 초기 탐색 단계에서 유용하게 활용될 수 있습니다.

In [ ]:
#

import ast

# 조건 키워드
keywords = ['소특', '특별', '소부장', '소재부품기술', '소재부품산업기술']

def match_keywords_in_project_name(row):
    try:
        obj = ast.literal_eval(row) if isinstance(row, str) else row
        matched_names = set()

        for proj in obj:
            proj_name_dict = proj.get('proj_명칭', {})
            사업명 = str(proj_name_dict.get('사업명', ''))
            if any(keyword in 사업명 for keyword in keywords):
                matched_names.add(사업명)

        return list(matched_names) if matched_names else None
    except:
        return None

# 1. 각 행에서 조건을 만족하는 사업명 추출
data['matched_사업명'] = data['project_data'].apply(match_keywords_in_project_name)

# 2. 결과가 존재하는 행만 추출
matched_df = data[data['matched_사업명'].notnull()]

# 3. 고유 사업명만 추출
from itertools import chain

# 여러 리스트를 하나로 합치기
all_matched_names = list(chain.from_iterable(matched_df['matched_사업명']))

# 고유한 사업명 집합 생성
unique_matched_names = set(all_matched_names)

# 4. 출력
print(f"조건을 만족하는 고유한 사업명 개수: {len(unique_matched_names)}")
print("고유 사업명 목록:")
for name in sorted(unique_matched_names):
    print(name)

조건을 만족하는 고유한 사업명 개수: 10
고유 사업명 목록:
소재부품기술개발
소재부품기술개발(R&D)
소재부품산업기술개발기반구축
소재부품산업기술개발기반구축(R&D)
중소기업기술혁신개발(소부장회계)
중소기업기술혁신개발(특별,R&D)
중소기업상용화기술개발(소부장회계)
중소기업상용화기술개발(특별,R&D)
창업성장기술개발(소부장회계)
창업성장기술개발(특별,R&D)


### **코드 해설**

#### **1. `match_keywords_in_project_name` 함수 정의**

*   **기능**: 주어진 `row` (일반적으로 `project_data` 컬럼의 한 셀 값)에서 각 프로젝트의 `'사업명'`을 확인하여, 미리 정의된 `keywords` 리스트에 있는 키워드 중 하나라도 포함하는 사업명을 추출합니다. 추출된 사업명들은 중복 없이 `set`에 저장된 후 리스트 형태로 반환됩니다. 일치하는 사업명이 없거나 오류 발생 시 `None`을 반환합니다.
*   **메서드/클래스/라이브러리**:
    *   `ast.literal_eval(row)`: 문자열로 표현된 Python 리터럴(리스트, 딕셔너리 등)을 실제 Python 객체로 안전하게 변환합니다.
    *   `isinstance(row, str)`: `row`가 문자열 타입인지 확인하여, 문자열인 경우에만 `ast.literal_eval`을 적용합니다.
    *   `dict.get(key, default_value)`: 딕셔너리에서 `key`에 해당하는 값을 가져오되, `key`가 없으면 `default_value`로 지정된 빈 딕셔너리 또는 빈 문자열을 반환하여 KeyError 발생을 방지합니다.
    *   `str(proj_name_dict.get('사업명', ''))`: 사업명 필드가 없거나 `None`일 경우를 대비해 빈 문자열로 처리하고 문자열로 변환합니다.
    *   `any(keyword in 사업명 for keyword in keywords)`: `keywords` 리스트의 각 `keyword`가 현재 `사업명` 문자열에 포함되어 있는지 확인합니다. 하나라도 `True`이면 전체는 `True`를 반환합니다.
    *   `set.add(item)`: `set`에 `item`을 추가합니다. `set`의 특성상 중복된 값은 한 번만 저장됩니다.
    *   `list(matched_names)`: `set`으로 수집된 사업명들을 다시 리스트로 변환합니다.
    *   `try-except` 블록: 데이터 파싱이나 처리 과정에서 발생할 수 있는 오류를 처리하고, 오류 발생 시 `None`을 반환합니다.

#### **2. `data['matched_사업명'] = data['project_data'].apply(match_keywords_in_project_name)`**

*   **기능**: 정의된 `match_keywords_in_project_name` 함수를 `data` DataFrame의 `project_data` 컬럼의 각 행에 적용하여, 키워드에 매칭되는 사업명 리스트를 새로운 컬럼 `matched_사업명`에 저장합니다.
*   **메서드/클래스/라이브러리**:
    *   `pandas.Series.apply(func)`: Pandas Series의 각 요소에 대해 지정된 함수(`func`)를 적용합니다.

#### **3. `matched_df = data[data['matched_사업명'].notnull()]`**

*   **기능**: `matched_사업명` 컬럼의 값이 `None`이 아닌 (즉, 키워드에 매칭되는 사업명이 하나라도 있는) 행들만 필터링하여 새로운 DataFrame `matched_df`를 생성합니다.
*   **메서드/클래스/라이브러리**:
    *   `pandas.Series.notnull()`: Series의 각 요소가 `null` (NaN, None 등)이 아닌지 여부를 `True`/`False`로 반환합니다.
    *   불리언 인덱싱 (`DataFrame[boolean_series]`): `True`/`False` 값을 가진 Series를 사용하여 DataFrame의 행을 선택합니다.

#### **4. `from itertools import chain` 및 고유 사업명 추출**

*   **기능**: `matched_df` DataFrame의 `matched_사업명` 컬럼은 각 행마다 리스트를 포함하고 있습니다. `chain.from_iterable`을 사용하여 이 모든 리스트를 하나의 평탄화된(flattened) 리스트로 만든 다음, `set()`을 통해 이들 중에서 중복을 제거한 고유한 사업명 집합을 생성합니다.
*   **메서드/클래스/라이브러리**:
    *   `itertools.chain.from_iterable(iterable)`: 여러 이터러블(여기서는 리스트들의 리스트)을 단일 이터러블로 연결합니다.
    *   `list()`: 이터러블을 리스트로 변환합니다.
    *   `set()`: 중복을 제거하여 고유한 요소들로 구성된 집합을 생성합니다.

#### **5. 결과 출력**

*   **기능**: `unique_matched_names`의 개수와 정렬된 고유 사업명 목록을 콘솔에 출력합니다.
*   **메서드/클래스/라이브러리**:
    *   `print()`: 값을 콘솔에 출력합니다.
    *   `len()`: 객체의 길이를 반환합니다 (여기서는 `set`의 크기).
    *   `sorted()`: 이터러블의 요소를 정렬된 새 리스트로 반환합니다.

#### **코드6**
#### 특정 정부 연구개발 사업에 대한 '신규' 과제를 식별

이 코드는 `data` DataFrame에서 특정 정부 연구개발 사업(미리 정의된 `target_projects` 목록)에 대한 '신규' 과제를 식별합니다. 이후 각 사업명과 그 세부 내역사업명, 연도별로 해당 과제에 참여한 고유 기업의 수를 집계하고, 이 정보를 MultiIndex를 가진 피벗 테이블 형태로 출력합니다. 이는 특정 R&D 정책이 어떤 사업명과 내역사업명으로, 어느 연도에 얼마나 많은 기업에 도달했는지 정량적으로 분석하는 데 활용될 수 있습니다.

In [ ]:
import pandas as pd
import ast
from itertools import chain

# 1. 타겟 사업명 목록 (정확히 일치해야 함)
target_projects = [
    '중소기업기술혁신개발(소부장회계)',
    '중소기업기술혁신개발(특별,R&D)',
    '중소기업기술혁신개발(일반,R&D)',
    '중소기업기술혁신개발(R&D)',
    '중소기업기술혁신개발',
    '중소기업상용화기술개발(소부장회계)',
    '중소기업상용화기술개발(특별,R&D)',
    '중소기업상용화기술개발(일반,R&D)',
    '중소기업상용화기술개발(R&D)',
    '중소기업상용화기술개발',
    '창업성장기술개발(소부장회계)',
    '창업성장기술개발(특별,R&D)',
    '창업성장기술개발(일반,R&D)',
    '창업성장기술개발(R&D)',
    '창업성장기술개발',
    '소재부품기술개발',
    '소재부품기술개발(R&D)',
    '소재부품산업기술개발기반구축(R&D)'
]

# 2. (id, year, 사업명, 내역사업명) 추출 함수
def extract_detailed_info(row):
    try:
        obj = ast.literal_eval(row['project_data']) if isinstance(row['project_data'], str) else row['project_data']
        matched = set()

        for proj in obj:
            proj_dict = proj.get('proj_명칭', {})
            proj_status = proj.get('proj_계속여부', {}).get('계속과제여부', '')
            사업명 = str(proj_dict.get('사업명', '')).strip()
            내역사업명 = str(proj_dict.get('내역사업명', '')).strip()

            # 조건: 타겟 사업명 중 하나 + '신규' 상태
            if 사업명 in target_projects and proj_status == '신규':
                matched.add((row['id'], row['year'], 사업명, 내역사업명))

        return list(matched) if matched else None
    except:
        return None

# 3. 모든 행에 대해 조건 필터링 및 추출 적용
matching_rows = data.apply(extract_detailed_info, axis=1)
flattened = list(chain.from_iterable([r for r in matching_rows if r]))

# 4. 데이터프레임 생성
matched_df = pd.DataFrame(flattened, columns=['id', 'year', '사업명', '내역사업명'])

# 5. groupby 및 고유 id 수 세기
grouped = matched_df.groupby(['사업명', '내역사업명', 'year'])['id'].nunique().reset_index()

# 6. 피벗 테이블 생성
pivot_table = grouped.pivot_table(index=['사업명', '내역사업명'], columns='year', values='id', fill_value=0).astype(int)

# 7. 결과 출력 및 저장
print(pivot_table)

pivot_table.to_csv("사업명_내역사업명_연도별_ID수.csv", encoding='utf-8-sig')


year                                                             2017  2018  \
사업명              내역사업명                                                        
소재부품기술개발         융복합소재부품개발                                         34     0   
                 이종기술융합형                                            0     0   
                 패키지형                                               0     0   
                 핵심소재경쟁력강화                                         15     0   
소재부품기술개발(R&D)    소재부품이종기술융합형                                        0    12   
...                                                               ...   ...   
창업성장기술개발(일반,R&D) 2021년도 창업성장기술개발사업 전략형 과제(4IR) 제1차 시행계획 공고          0     0   
                 2021년도 창업성장기술개발사업 전략형(4차산업혁명) 제2차 시행계획 공고          0     0   
창업성장기술개발(특별,R&D) 2021년도 창업성장기술개발사업 소부장 스타트업 100 연계과제 시행계획 공고        0     0   
                 2021년도 창업성장기술개발사업 전략형 과제(소재·부품·장비) 제1차 시행계획 공고     0     0   
                 2021년도 창업성장기술개발사업 전략형(소재·부품·장비) 제2차

### **코드 해설**

#### **1. `target_projects` 목록 정의**

*   **기능**: 코드가 식별하고자 하는 특정 정부 연구개발 사업명의 정확한 목록을 정의합니다. 이 목록에 포함된 사업명만 분석 대상이 됩니다.
*   **메서드/클래스/라이브러리**: Python의 내장 `list` 데이터 타입입니다.

#### **2. `extract_detailed_info` 함수 정의**

*   **기능**: `data` DataFrame의 각 행에서 `project_data` 컬럼을 분석하여, `target_projects` 목록에 있고 `'계속과제여부'`가 '신규'인 프로젝트의 `(id, year, 사업명, 내역사업명)` 튜플을 추출합니다. 중복을 방지하기 위해 `set`을 사용하고, 최종적으로 리스트 형태로 반환합니다. 조건에 맞는 프로젝트가 없거나 오류 발생 시 `None`을 반환합니다.
*   **메서드/클래스/라이브러리**:
    *   `ast.literal_eval(row['project_data'])`: 문자열로 저장된 Python 리터럴(리스트 형태)을 실제 Python 객체로 안전하게 변환합니다.
    *   `isinstance(obj, str)`: 객체가 문자열인지 확인합니다.
    *   `dict.get(key, default_value)`: 딕셔너리에서 키에 해당하는 값을 가져오되, 키가 없으면 기본값을 반환하여 오류를 방지합니다.
    *   `str()`: 객체를 문자열로 변환합니다.
    *   `proj_name in target_projects`: `proj_name`이 `target_projects` 목록에 포함되어 있는지 확인합니다.
    *   `set.add(item)`: `set`에 요소를 추가합니다. `set`은 중복을 허용하지 않습니다.
    *   `list(matched)`: `set`을 다시 리스트로 변환합니다.
    *   `try-except` 블록: 데이터 처리 중 발생할 수 있는 예외를 처리합니다.

#### **3. 모든 행에 대해 조건 필터링 및 추출 적용 (`matching_rows`, `flattened` 생성)**

*   **기능**: `extract_detailed_info` 함수를 `data` DataFrame의 각 행에 적용하여 각 행별로 조건에 맞는 프로젝트 정보 리스트를 담은 Series `matching_rows`를 생성하고, 이를 `itertools.chain.from_iterable`을 사용하여 하나의 단일 평탄화된 리스트 `flattened`로 결합합니다.
*   **메서드/클래스/라이브러리**:
    *   `pandas.DataFrame.apply(func, axis=1)`: DataFrame의 각 행(`axis=1`)에 함수(`func`)를 적용합니다.
    *   `itertools.chain.from_iterable(iterable)`: 여러 이터러블(여기서는 리스트들의 리스트)을 단일 이터러블로 연결하는 `itertools` 모듈의 함수입니다.

#### **4. 데이터프레임 생성 (`matched_df`)**

*   **기능**: 평탄화된 리스트 `flattened`를 사용하여 `id`, `year`, `사업명`, `내역사업명` 컬럼을 가진 새로운 Pandas DataFrame `matched_df`를 생성합니다.
*   **메서드/클래스/라이브러리**:
    *   `pandas.DataFrame(data, columns=...)`: 데이터와 컬럼 이름을 지정하여 DataFrame을 생성합니다.

#### **5. `groupby`를 이용한 고유 `id` 수 집계 (`grouped` 생성)**

*   **기능**: `matched_df`를 `'사업명'`, `'내역사업명'`, `'year'` 기준으로 그룹화한 후, 각 그룹별로 고유한 `'id'`의 수를 계산합니다. `reset_index()`를 사용하여 그룹화된 결과를 일반 DataFrame 형태로 변환합니다.
*   **메서드/클래스/라이브러리**:
    *   `pandas.DataFrame.groupby([...])`: 지정된 컬럼들을 기준으로 데이터를 그룹화합니다.
    *   `pandas.core.groupby.DataFrameGroupBy['id'].nunique()`: 각 그룹 내에서 `'id'` 컬럼의 고유한 값의 수를 계산합니다.
    *   `pandas.DataFrame.reset_index()`: 그룹화 후 생성된 MultiIndex를 일반 컬럼으로 변환합니다.

#### **6. 피벗 테이블 형태로 결과 출력 (`pivot_table` 생성 및 출력)**

*   **기능**: `grouped` DataFrame을 사용하여 `['사업명', '내역사업명']`을 인덱스로, `'year'`를 컬럼으로, 그리고 각 셀에 해당 연도의 고유 `'id'` 수를 나타내는 피벗 테이블을 생성합니다. `fill_value=0`로 결측값을 0으로 채우고, `astype(int)`로 정수형으로 변환합니다. 최종적으로 이 피벗 테이블을 콘솔에 출력합니다.
*   **메서드/클래스/라이브러리**:
    *   `pandas.DataFrame.pivot_table(index=..., columns=..., values=..., fill_value=...)`: DataFrame을 피벗하여 새로운 형태의 DataFrame을 생성합니다. `pivot` 함수와 달리 `pivot_table`은 집계 함수를 기본적으로 사용할 수 있으며 `index`에 여러 컬럼을 지정할 수 있습니다.
    *   `pandas.DataFrame.fillna(value)`: DataFrame의 결측값(NaN)을 지정된 값으로 채웁니다.
    *   `pandas.DataFrame.astype(dtype)`: DataFrame의 데이터 타입을 변경합니다.
    *   `print()`: 값을 콘솔에 출력합니다.

#### **코드6.1**
#### 특정 정부 연구개발 사업에 대한 신규 과제를 식별

이 코드는 `data` DataFrame에서 특정 정부 연구개발 사업(예: '중소기업기술혁신개발', '소재부품기술개발' 등 미리 정의된 목록)에 대한 신규 과제를 식별합니다. 이후, 각 사업명과 연도별로 해당 신규 과제에 참여한 고유 기업의 수를 집계하고, 이 정보를 시각적으로 파악하기 쉬운 피벗 테이블 형태로 정리하여 출력합니다. 이 과정을 통해 특정 R&D 정책 지원이 어떤 사업명으로, 어느 연도에 얼마나 많은 기업에 도달했는지 정량적으로 분석할 수 있습니다.

In [ ]:
import pandas as pd
import ast

# 1. 정확히 일치해야 할 사업명 키워드 목록
target_projects = [
    '중소기업기술혁신개발(소부장회계)',
    '중소기업기술혁신개발(특별,R&D)',
    '중소기업기술혁신개발(일반,R&D)',
    '중소기업기술혁신개발(R&D)',
    '중소기업기술혁신개발',
    '중소기업상용화기술개발(소부장회계)',
    '중소기업상용화기술개발(특별,R&D)',
    '중소기업상용화기술개발(일반,R&D)',
    '중소기업상용화기술개발(R&D)',
    '중소기업상용화기술개발',
    '창업성장기술개발(소부장회계)',
    '창업성장기술개발(특별,R&D)',
    '창업성장기술개발(일반,R&D)',
    '창업성장기술개발(R&D)',
    '창업성장기술개발',
    '소재부품기술개발',
    '소재부품기술개발(R&D)',
    '소재부품산업기술개발기반구축(R&D)'
]

# 2. 조건에 맞는 (id, year, 사업명) 세트를 추출하는 함수
def extract_matching_project_info(row):
    try:
        obj = ast.literal_eval(row['project_data']) if isinstance(row['project_data'], str) else row['project_data']
        matched = set()

        for proj in obj:
            proj_name = str(proj.get('proj_명칭', {}).get('사업명', ''))
            proj_status = str(proj.get('proj_계속여부', {}).get('계속과제여부', ''))

            # 조건: 정확한 사업명 + '신규'
            if proj_name in target_projects and proj_status == '신규':
                matched.add((row['id'], row['year'], proj_name))

        return list(matched) if matched else None
    except Exception as e:
        return None

# 3. 데이터프레임에서 조건 만족하는 (id, year, 사업명) 추출
matching_rows = data.apply(extract_matching_project_info, axis=1)

# 4. None이 아닌 리스트만 모아서 평탄화(flatten)
from itertools import chain

flattened = list(chain.from_iterable([r for r in matching_rows if r]))

# 5. 결과를 데이터프레임으로 변환
matched_df = pd.DataFrame(flattened, columns=['id', 'year', '사업명'])

# 6. groupby로 고유 id 수 세기
result = matched_df.groupby(['사업명', 'year'])['id'].nunique().reset_index()

# 7. 피벗 테이블 형태로 출력
pivot_table = result.pivot(index='사업명', columns='year', values='id').fillna(0).astype(int)

# 8. 결과 출력
print(pivot_table)


year                 2017  2018  2019  2020  2021  2022
사업명                                                    
소재부품기술개발               47     0     0     0     0    45
소재부품기술개발(R&D)           0    21    26   130   112     0
소재부품산업기술개발기반구축(R&D)     0     0     0    76    76    48
중소기업기술혁신개발            211     0     0     0     0   132
중소기업기술혁신개발(R&D)         0   122    78   338     0     0
중소기업기술혁신개발(소부장회계)       0     0     0     0     0    67
중소기업기술혁신개발(일반,R&D)      0     0     0     0    52     0
중소기업기술혁신개발(특별,R&D)      0     0     0     0    74     0
중소기업상용화기술개발           148     0     0     0     0    90
중소기업상용화기술개발(R&D)        0    96   119   137     0     0
중소기업상용화기술개발(소부장회계)      0     0     0     0     0    14
중소기업상용화기술개발(일반,R&D)     0     0     0     0    33     0
중소기업상용화기술개발(특별,R&D)     0     0     0     0     9     0
창업성장기술개발               60     0     0     0     0    31
창업성장기술개발(R&D)           0   105   185   176     0     0
창업성장기술개발(소부장회계)         0     0     0     0     

### **코드 해설**

#### **1. `target_projects` 목록 정의**

*   **기능**: 코드가 식별하고자 하는 특정 정부 연구개발 사업명의 정확한 목록을 정의합니다. 이 목록에 포함된 사업명만 분석 대상이 됩니다.
*   **메서드/클래스/라이브러리**: Python의 내장 `list` 데이터 타입입니다.

#### **2. `extract_matching_project_info` 함수 정의**

*   **기능**: `data` DataFrame의 각 행에서 `project_data` 컬럼을 분석하여, `target_projects` 목록에 있고 `'계속과제여부'`가 '신규'인 프로젝트의 `(id, year, 사업명)` 튜플을 추출합니다. 중복을 방지하기 위해 `set`을 사용하고, 최종적으로 리스트 형태로 반환합니다. 조건에 맞는 프로젝트가 없거나 오류 발생 시 `None`을 반환합니다.
*   **메서드/클래스/라이브러리**:
    *   `ast.literal_eval(row['project_data'])`: 문자열로 저장된 Python 리터럴(리스트 형태)을 실제 Python 객체로 안전하게 변환합니다.
    *   `isinstance(obj, str)`: 객체가 문자열인지 확인합니다.
    *   `dict.get(key, default_value)`: 딕셔너리에서 키에 해당하는 값을 가져오되, 키가 없으면 기본값을 반환하여 오류를 방지합니다.
    *   `str()`: 객체를 문자열로 변환합니다.
    *   `proj_name in target_projects`: `proj_name`이 `target_projects` 목록에 포함되어 있는지 확인합니다.
    *   `set.add(item)`: `set`에 요소를 추가합니다. `set`은 중복을 허용하지 않습니다.
    *   `list(matched)`: `set`을 다시 리스트로 변환합니다.
    *   `try-except` 블록: 데이터 처리 중 발생할 수 있는 예외를 처리합니다.

#### **3. 조건 만족하는 `(id, year, 사업명)` 추출 (`matching_rows` 생성)**

*   **기능**: `extract_matching_project_info` 함수를 `data` DataFrame의 각 행에 적용하여, 각 행별로 조건에 맞는 프로젝트 정보 리스트를 담은 Series `matching_rows`를 생성합니다.
*   **메서드/클래스/라이브러리**:
    *   `pandas.DataFrame.apply(func, axis=1)`: DataFrame의 각 행(`axis=1`)에 함수(`func`)를 적용합니다.

#### **4. 리스트 평탄화 (`flattened` 생성)**

*   **기능**: `matching_rows` Series에는 여러 리스트가 포함될 수 있습니다. `itertools.chain.from_iterable`을 사용하여 이 모든 리스트를 하나의 단일 평탄화된 리스트 `flattened`로 결합합니다.
*   **메서드/클래스/라이브러리**:
    *   `itertools.chain.from_iterable(iterable)`: 여러 이터러블(여기서는 리스트들의 리스트)을 단일 이터러블로 연결하는 `itertools` 모듈의 함수입니다.

#### **5. 결과 데이터프레임 생성 (`matched_df`)**

*   **기능**: 평탄화된 리스트 `flattened`를 사용하여 `id`, `year`, `사업명` 컬럼을 가진 새로운 Pandas DataFrame `matched_df`를 생성합니다.
*   **메서드/클래스/라이브러리**:
    *   `pandas.DataFrame(data, columns=...)`: 데이터와 컬럼 이름을 지정하여 DataFrame을 생성합니다.

#### **6. `groupby`를 이용한 고유 `id` 수 집계 (`result` 생성)**

*   **기능**: `matched_df`를 `'사업명'`과 `'year'` 기준으로 그룹화한 후, 각 그룹별로 고유한 `'id'`의 수를 계산합니다. `reset_index()`를 사용하여 그룹화된 결과를 일반 DataFrame 형태로 변환합니다.
*   **메서드/클래스/라이브러리**:
    *   `pandas.DataFrame.groupby([...])`: 지정된 컬럼들을 기준으로 데이터를 그룹화합니다.
    *   `pandas.core.groupby.DataFrameGroupBy['id'].nunique()`: 각 그룹 내에서 `'id'` 컬럼의 고유한 값의 수를 계산합니다.
    *   `pandas.DataFrame.reset_index()`: 그룹화 후 생성된 MultiIndex를 일반 컬럼으로 변환합니다.

#### **7. 피벗 테이블 형태로 결과 출력 (`pivot_table` 생성 및 출력)**

*   **기능**: `result` DataFrame을 사용하여 `'사업명'`을 인덱스로, `'year'`를 컬럼으로, 그리고 각 셀에 해당 연도의 고유 `'id'` 수를 나타내는 피벗 테이블을 생성합니다. `fillna(0)`로 결측값을 0으로 채우고, `astype(int)`로 정수형으로 변환합니다. 최종적으로 이 피벗 테이블을 콘솔에 출력합니다.
*   **메서드/클래스/라이브러리**:
    *   `pandas.DataFrame.pivot(index=..., columns=..., values=...)`: DataFrame을 피벗하여 새로운 형태의 DataFrame을 생성합니다.
    *   `pandas.DataFrame.fillna(value)`: DataFrame의 결측값(NaN)을 지정된 값으로 채웁니다.
    *   `pandas.DataFrame.astype(dtype)`: DataFrame의 데이터 타입을 변경합니다.
    *   `print()`: 값을 콘솔에 출력합니다.

#### **코드7**
#### 특정 정부 연구개발 사업에 대한 '신규' 과제를 식별하고 참여 기업 수 집계

이 코드는 `data` DataFrame에서 특정 정부 연구개발 사업(미리 정의된 `target_projects` 목록)에 대한 '신규' 과제를 식별합니다. 이후, 다음 두 가지 결과를 생성하고 저장합니다.

1.  **요약 피벗 테이블**: 각 사업명과 연도별로 해당 신규 과제에 참여한 고유 기업의 수를 집계하여 피벗 테이블 형태로 CSV 파일(`사업명별_연도별_ID수_요약.csv`)로 저장합니다.
2.  **상세 과제명 목록**: 특정 사업명에 해당하는 신규 과제의 ID, 연도, 사업명, 그리고 구체적인 과제명을 포함하는 상세 목록을 CSV 파일(`신규_사업명별_과제명_목록.csv`)로 저장합니다.

이 과정을 통해 특정 R&D 정책 지원이 어떤 사업명으로, 어느 연도에 얼마나 많은 기업에 도달했으며, 어떤 구체적인 과제들이 수행되었는지 정량적으로 분석할 수 있는 데이터를 제공합니다.

In [ ]:
import pandas as pd
import ast
from itertools import chain

# 1. 정확히 일치해야 할 사업명 키워드 목록
target_projects = [
    '중소기업기술혁신개발(소부장회계)',
    '중소기업기술혁신개발(특별,R&D)',
    '중소기업상용화기술개발(소부장회계)',
    '중소기업상용화기술개발(특별,R&D)',
    '창업성장기술개발(소부장회계)',
    '창업성장기술개발(특별,R&D)',
    '소재부품기술개발',
    '소재부품기술개발(R&D)',
    '소재부품산업기술개발기반구축(R&D)'
]

# 2. (id, year, 사업명) & 과제명 추출 함수
def extract_info(row):
    try:
        obj = ast.literal_eval(row['project_data']) if isinstance(row['project_data'], str) else row['project_data']
        matched_project_set = set()
        project_detail_list = []

        for proj in obj:
            proj_name = str(proj.get('proj_명칭', {}).get('사업명', ''))
            proj_status = str(proj.get('proj_계속여부', {}).get('계속과제여부', ''))
            task_name = str(proj.get('proj_명칭', {}).get('과제명', '')).strip()

            if proj_name in target_projects and proj_status == '신규':
                key_tuple = (row['id'], row['year'], proj_name)
                matched_project_set.add(key_tuple)

                # 과제명 포함 저장
                if task_name:  # 빈 값은 제외
                    project_detail_list.append((row['id'], row['year'], proj_name, task_name))

        return list(matched_project_set), project_detail_list

    except Exception as e:
        return None, None

# 3. 전체 행에 대해 적용
matched_project_tuples = []
project_task_rows = []

for _, row in data.iterrows():
    matched, detailed = extract_info(row)
    if matched:
        matched_project_tuples.extend(matched)
    if detailed:
        project_task_rows.extend(detailed)

# 4. 결과 DataFrame 생성

## (1) 요약용 pivot 테이블: id 개수
df_summary = pd.DataFrame(matched_project_tuples, columns=['id', 'year', '사업명'])
grouped = df_summary.groupby(['사업명', 'year'])['id'].nunique().reset_index()
pivot_table = grouped.pivot(index='사업명', columns='year', values='id').fillna(0).astype(int)

## (2) 보조 테이블: 과제명 포함
df_task = pd.DataFrame(project_task_rows, columns=['id', 'year', '사업명', '과제명'])

# 5. CSV 저장
pivot_table.to_csv("사업명별_연도별_ID수_요약.csv", encoding='utf-8-sig')
df_task.to_csv("신규_사업명별_과제명_목록.csv", index=False, encoding='utf-8-sig')

# 6. 확인 출력
print("✅ CSV 저장 완료")
print("👉 요약 파일: 사업명별_연도별_ID수_요약.csv")
print("👉 과제명 파일: 신규_사업명별_과제명_목록.csv")


### **코드 해설**

#### **1. `target_projects` 목록 정의**

*   **기능**: 코드가 식별하고자 하는 특정 정부 연구개발 사업명의 정확한 목록을 정의합니다. 이 목록에 포함된 사업명만 분석 대상이 됩니다.
*   **메서드/클래스/라이브러리**: Python의 내장 `list` 데이터 타입입니다.

#### **2. `extract_info` 함수 정의**

*   **기능**: `data` DataFrame의 각 행에서 `project_data` 컬럼을 분석하여, `target_projects` 목록에 있고 `'계속과제여부'`가 '신규'인 프로젝트의 `(id, year, 사업명)` 튜플과 `(id, year, 사업명, 과제명)` 튜플을 추출합니다. 중복을 방지하기 위해 `set`을 사용하며, 최종적으로 리스트 형태로 반환합니다. 조건에 맞는 프로젝트가 없거나 오류 발생 시 `None`을 반환합니다.
*   **메서드/클래스/라이브러리**:
    *   `ast.literal_eval(row['project_data'])`: 문자열로 저장된 Python 리터럴(리스트 형태)을 실제 Python 객체로 안전하게 변환합니다.
    *   `isinstance(obj, str)`: 객체가 문자열인지 확인합니다.
    *   `dict.get(key, default_value)`: 딕셔너리에서 키에 해당하는 값을 가져오되, 키가 없으면 기본값을 반환하여 오류를 방지합니다.
    *   `str()`: 객체를 문자열로 변환합니다.
    *   `proj_name in target_projects`: `proj_name`이 `target_projects` 목록에 포함되어 있는지 확인합니다.
    *   `set.add(item)`: `set`에 요소를 추가합니다. `set`은 중복을 허용하지 않습니다.
    *   `list(matched_project_set)`: `set`을 다시 리스트로 변환합니다.
    *   `project_detail_list.append()`: 과제명을 포함하는 상세 리스트에 요소를 추가합니다.
    *   `try-except` 블록: 데이터 처리 중 발생할 수 있는 예외를 처리합니다.

#### **3. 전체 행에 대해 적용 (`matched_project_tuples`, `project_task_rows` 생성)**

*   **기능**: `extract_info` 함수를 `data` DataFrame의 각 행에 적용하여 조건에 맞는 프로젝트 정보와 과제명을 추출하고, 이를 각각 `matched_project_tuples` 리스트와 `project_task_rows` 리스트에 평탄화하여 저장합니다.
*   **메서드/클래스/라이브러리**:
    *   `pandas.DataFrame.iterrows()`: DataFrame의 각 행을 (인덱스, Series) 쌍으로 순회합니다.
    *   `list.extend()`: 리스트의 끝에 다른 리스트의 모든 항목을 추가합니다.

#### **4. 결과 DataFrame 생성 (`df_summary`, `df_task`)**

*   **기능**: 추출된 `matched_project_tuples`와 `project_task_rows`를 각각 새로운 Pandas DataFrame `df_summary`와 `df_task`로 변환합니다.
*   **메서드/클래스/라이브러리**:
    *   `pandas.DataFrame(data, columns=...)`: 데이터와 컬럼 이름을 지정하여 DataFrame을 생성합니다.

#### **5. `groupby`를 이용한 고유 `id` 수 집계 및 피벗 테이블 생성 (`grouped`, `pivot_table`)**

*   **기능**: `df_summary`를 `'사업명'`과 `'year'` 기준으로 그룹화한 후, 각 그룹별로 고유한 `'id'`의 수를 계산합니다. 이 결과를 피벗하여 `'사업명'`을 인덱스로, `'year'`를 컬럼으로, 고유 `'id'` 수를 값으로 하는 피벗 테이블을 생성합니다. 결측값은 0으로 채우고 정수형으로 변환합니다.
*   **메서드/클래스/라이브러리**:
    *   `pandas.DataFrame.groupby([...])`: 지정된 컬럼들을 기준으로 데이터를 그룹화합니다.
    *   `pandas.core.groupby.DataFrameGroupBy['id'].nunique()`: 각 그룹 내에서 `'id'` 컬럼의 고유한 값의 수를 계산합니다.
    *   `pandas.DataFrame.reset_index()`: 그룹화 후 생성된 MultiIndex를 일반 컬럼으로 변환합니다.
    *   `pandas.DataFrame.pivot(index=..., columns=..., values=...)`: DataFrame을 피벗하여 새로운 형태의 DataFrame을 생성합니다.
    *   `pandas.DataFrame.fillna(value)`: DataFrame의 결측값(NaN)을 지정된 값으로 채웁니다.
    *   `pandas.DataFrame.astype(dtype)`: DataFrame의 데이터 타입을 변경합니다.

#### **6. CSV 저장 및 확인 출력**

*   **기능**: 생성된 `pivot_table`과 `df_task`를 각각 CSV 파일로 저장하고, 저장 완료 메시지를 출력합니다.
*   **메서드/클래스/라이브러리**:
    *   `DataFrame.to_csv()`: DataFrame을 CSV 파일로 저장합니다. `index=False`는 DataFrame 인덱스를 CSV에 포함하지 않도록 하고, `encoding='utf-8-sig'`는 한글 깨짐을 방지하는 인코딩 방식입니다。
    *   `print()`: 값을 콘솔에 출력합니다.

#### **코드8**
#### 조건을 만족하는 행에 special 값 부여

이 코드는 `data` DataFrame을 분석하여, 미리 정의된 특정 내역사업명(`special_keywords`)에 해당하며 '신규' 과제로 분류된 정부 연구개발 프로젝트에 참여한 기업들을 식별합니다. 그리고 각 기업(`id`)이 이러한 조건을 처음으로 만족한 `year`를 찾아 `data` DataFrame의 새로운 컬럼인 `'special'`에 기록합니다. 이는 특정 정책이나 사업의 시작 연도를 기준으로 기업들을 그룹화하거나 정책 효과를 분석하기 위한 전처리 단계에서 유용하게 활용될 수 있습니다.

In [ ]:
import pandas as pd
import ast
from collections import defaultdict

# 1. 내역사업명 키워드 목록 (정확 일치)
special_keywords = set([
    '2020년도 중소기업기술혁신개발사업 ‘시장대응형 과제’ 제2차 시행계획 공고',
    '소재부품기술기반혁신',
    '2020년도 중소기업기술혁신개발사업 \'시장대응형 과제\' 제1차 시행계획 수정공고',
    '소재부품패키지형',
    '2020년도 구매조건부신제품개발사업 구매연계형 과제 자유응모(2차) 시행계획 공고',
    '2020년도 중소기업기술혁신개발사업 ‘시장확대형 과제’ 제2차 시행계획 수정공고',
    '2020년도 중소기업기술혁신개발사업 ‘시장확대형 과제’ 제1차 시행계획 수정공고',
    '전략핵심소재자립화기술개발',
    '2020년도 구매조건부신제품개발사업 구매연계형 과제 자유응모(1차) 시행계획 공고',
    '소재부품이종기술융합형',
    '2020년도 창업성장기술개발사업 \'전략형 창업과제(소재·부품·장비)\' 제2차 시행계획 수정 공고',
    '2020년도 창업성장기술개발사업 ’전략형 창업과제(소재·부품·장비)‘ 제1차 시행계획 공고',
    '2020년도 중소기업기술혁신개발사업 시장확대형 과제 제4차 시행계획 공고(후불형과제)',
    '2020년도 구매조건부신제품개발사업 공동투자형 과제 자유응모(4차) 시행계획 공고',
    '2020년도 중소기업기술혁신개발사업 ‘시장확대형 과제’ 제3차 시행계획 공고(비대면 분야)',
    '2020년도 구매조건부신제품개발사업 공동투자형 과제 자유응모(7차) 시행계획 공고',
    '2020년도 구매조건부신제품개발사업 공동투자형 과제 자유응모(6차) 시행계획 공고',
    '2020년도 구매조건부신제품개발사업 공동투자형 과제 자유응모(3차) 시행계획 공고',
    '2020년도 구매조건부신제품개발사업 공동투자형 과제 자유응모(5차) 시행계획 공고',
    '2020년도 구매조건부신제품개발사업 공동투자형 과제 자유응모(2차) 시행계획 공고',
    '2020년도 구매조건부신제품개발사업 구매연계형 과제 지정공모(2차) 시행계획 공고',
    '이종기술융합형',
    '패키지형',
    '소재부품기술기반혁신사업',
    '2019년도 중소기업 기술혁신개발사업(소재·부품·장비분야 추경사업)',
    '2022년도 중소기업기술혁신개발사업 \'강소기업100\' 과제 시행계획 공고',
    '2022년도 중소기업기술혁신개발사업 \'강소기업100\' 과제 시행계획(2차) 공고',
    '2022년도 중소기업기술혁신개발사업 \'소부장전략\' 과제 시행계획 공고',
    '2022년도 중소기업기술혁신개발사업 ‘소부장일반(일반과제)’ 상반기 시행계획 공고',
    '2022년도 중소기업기술혁신개발사업 ‘소부장일반(일반과제)’ 하반기 시행계획 공고',
    '2022년도 중소기업기술혁신개발사업 ‘소부장일반(재도약과제)’ 상반기 시행계획 공고',
    '2022년도 중소기업기술혁신개발사업 소부장전략(함께달리기) 시행계획 공고',
    '2022년도 중소기업기술혁신개발사업(소부장전략) 대중소기업 상생모델 추천기업 사업계획서 접수 안내',
    '사업연계형(소부장 전략)',
    '중소기업기술혁신개발사업(소부장일반)(22년 컨소시엄형R&D 선정기업 대상)',
    '중소기업기술혁신개발사업(소부장전략)(22년 컨소시엄형R&D 선정기업 대상)',
    '2021년 중소기업기술혁신개발사업 \'강소기업100\' 과제 제1차 시행계획 공고',
    '2021년 중소기업기술혁신개발사업 \'소부장일반\' 과제 제1차 시행계획 수정 공고',
    '2021년 중소기업기술혁신개발사업 \'소부장전략 과제\' 제1차 시행계획 수정 공고',
    '2021년 중소기업기술혁신개발사업 소부장전략 과제 2차 시행계획 공고',
    '2021년도 중소기업기술혁신개발사업 강소기업100 과제 2차 시행계획 공고',
    '2021년도 중소기업기술혁신개발사업 소부장일반 과제 제 2차 시행계획 공고',
    '2021년도 중소기업기술혁신개발사업 소부장전략과제(함께달리기) 시행계획 공고',
    '2022년도 중소기업 구매조건부신제품개발사업 \'공동투자형 과제\' 자유공모(2차) 시행계획 공고',
    '2022년도 중소기업 구매조건부신제품개발사업 \'구매연계형 과제\' 자유공모(2차) 시행계획 공고',
    '2022년도 중소기업 구매조건부신제품개발사업 ‘공동투자형 과제’ 자유공모(1차) 시행계획  공고',
    '2022년도 중소기업 구매조건부신제품개발사업 ‘구매연계형 과제’ 자유공모(1차) 시행계획 공고',
    '2021년도 구매조건부신제품개발사업 공동투자형 과제 자유응모(2차) 시행계획 공고',
    '2021년도 구매조건부신제품개발사업 구매연계형 과제 자유응모(1차) 시행계획 공고',
    '2021년도 구매조건부신제품개발사업 구매연계형 과제 자유응모(2차) 시행계획 공고',
    '2022년도 창업성장기술개발사업 소재·부품·장비 스타트업100 연계과제 시행계획 공고',
    '2022년도 창업성장기술개발사업 전략형(소재부품장비) 제1차 시행계획 공고',
    '2021년도 창업성장기술개발사업 소부장 스타트업 100 연계과제 시행계획 공고',
    '2021년도 창업성장기술개발사업 전략형 과제(소재·부품·장비) 제1차 시행계획 공고',
    '2021년도 창업성장기술개발사업 전략형(소재·부품·장비) 제2차 시행계획 공고'
])

# 2. id별 special year를 저장
id_to_special = defaultdict(set)

# 3. 데이터 처리
for _, row in data.iterrows():
    try:
        projects = ast.literal_eval(row['project_data']) if isinstance(row['project_data'], str) else row['project_data']
        for proj in projects:
            내역사업명 = proj.get('proj_명칭', {}).get('내역사업명', '').strip()
            계속과제여부 = proj.get('proj_계속여부', {}).get('계속과제여부', '').strip()
            if 내역사업명 in special_keywords and 계속과제여부 == '신규':
                id_to_special[row['id']].add(row['year'])
    except Exception as e:
        continue  # malformed row, skip

# 4. special 컬럼 생성 (각 id별 최초 year 할당)
def assign_special(id_):
    years = id_to_special.get(id_, set())
    if not years:
        return None
    if len(years) > 1:
        print(f"⚠️ ID '{id_}' has multiple special years: {sorted(years)}")
    return min(years)  # 선택 기준은 earliest year (or could use max/year string)

data['special'] = data['id'].apply(assign_special)


### **코드 해설**

#### **1. `special_keywords` 목록 정의**

*   **기능**: 분석 대상이 되는 특정 내역사업명들을 포함하는 `set` (집합)을 정의합니다. `set`을 사용함으로써 멤버십 검사(`in` 연산)가 매우 효율적입니다.
*   **메서드/클래스/라이브러리**: Python의 내장 `set` 데이터 타입입니다.

#### **2. `id_to_special` 초기화**

*   **기능**: 각 `id` (기업)별로 조건에 해당하는 `year`들을 저장할 딕셔너리를 초기화합니다. `defaultdict(set)`을 사용하면 키가 없을 때 자동으로 빈 `set`을 생성하여 `add` 메서드를 바로 사용할 수 있게 합니다.
*   **메서드/클래스/라이브러리**:
    *   `collections.defaultdict`: `collections` 모듈의 클래스로, 딕셔너리에서 존재하지 않는 키에 접근할 때 기본값을 자동으로 생성해줍니다. 여기서는 기본값이 `set`으로 설정되어 있습니다.

#### **3. 데이터 처리 루프**

*   **기능**: `data` DataFrame의 각 행을 순회하면서 `project_data` 컬럼 내의 프로젝트 정보를 분석합니다. 각 프로젝트의 `'내역사업명'`이 `special_keywords`에 포함되고 `'계속과제여부'`가 '신규'인 경우, 해당 기업의 `id`와 `year`를 `id_to_special` 딕셔너리에 추가합니다.
*   **메서드/클래스/라이브러리**:
    *   `data.iterrows()`: Pandas DataFrame의 각 행을 (인덱스, Series) 쌍으로 순회합니다.
    *   `ast.literal_eval(row['project_data'])`: `project_data`가 문자열 형태로 저장되어 있을 경우, 이를 실제 Python 객체(리스트)로 안전하게 변환합니다.
    *   `isinstance(obj, str)`: 객체가 문자열인지 확인합니다.
    *   `proj.get('key', {})` 또는 `proj_name.get('key', '')`: 딕셔너리에서 키에 해당하는 값을 가져오되, 키가 없을 경우 기본값(빈 딕셔너리 또는 빈 문자열)을 반환하여 `KeyError` 발생을 방지합니다.
    *   `.strip()`: 문자열 양 끝의 공백을 제거합니다.
    *   `id_to_special[row['id']].add(row['year'])`: 조건에 맞는 프로젝트를 발견하면 해당 기업(`id`)의 `year`를 `id_to_special` 딕셔너리에 추가합니다. `defaultdict(set)` 덕분에 `add` 메서드를 바로 사용할 수 있습니다.
    *   `try-except` 블록: `ast.literal_eval` 또는 데이터 접근 중 발생할 수 있는 예외를 처리하여, 해당 행은 건너뛰고 다음 행으로 진행합니다.

#### **4. `special` 컬럼 생성**

*   **기능**: `assign_special` 함수를 정의하여 각 기업(`id`)별로 `id_to_special`에 저장된 연도들 중 가장 빠른 연도를 반환합니다. 이 함수를 `data['id']` 컬럼에 적용하여 `data` DataFrame에 새로운 `'special'` 컬럼을 생성합니다.
*   **메서드/클래스/라이브러리**:
    *   `assign_special(id_)`: 각 기업 `id`를 인자로 받아, 해당 기업이 조건에 맞는 프로젝트를 수행한 연도(`years`)를 찾고, `min(years)`를 통해 가장 이른 연도를 반환합니다.
    *   `id_to_special.get(id_, set())`: `id_to_special` 딕셔너리에서 `id_`에 해당하는 연도 `set`을 가져오거나, 없으면 빈 `set`을 반환합니다.
    *   `len(years) > 1`: 여러 해에 걸쳐 조건에 맞는 프로젝트를 수행한 기업이 있을 경우 경고 메시지를 출력합니다.
    *   `data['id'].apply(assign_special)`: Pandas Series의 `apply` 메서드를 사용하여 `assign_special` 함수를 `id` 컬럼의 모든 고유 값에 적용하고, 그 결과를 `'special'`이라는 새 컬럼에 할당합니다.